# 📘 Notebook 6 — Evaluation & Inference

**MiniMind Step-by-Step Series** | Milestone 6 of 6 🎉

In Notebooks 1-5, we built the tokenizer, model, data pipeline, trained, and fine-tuned with LoRA. In this final notebook, we explore **how to evaluate and use** our trained model — decoding strategies, temperature effects, perplexity, and multi-turn conversation.

**Learning Objectives:**
- Understand and compare decoding strategies (greedy, sampling, top-k, top-p/nucleus)
- Explore temperature's effect on generation diversity
- Implement perplexity evaluation
- Build a multi-turn conversational interface
- Compare pretrained vs SFT vs LoRA-merged models

In [ ]:
# === Recap: Complete code from Notebooks 1-5 (Config + Model + Tokenizer + Data + Pretrain + SFT + LoRA) ===
!pip install tokenizers torch transformers --quiet
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, json, os
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Config ---
@dataclass
class MiniMindConfig:
    hidden_size: int = 768; num_hidden_layers: int = 8; num_attention_heads: int = 8
    num_key_value_heads: int = 4; head_dim: int = 96; vocab_size: int = 6400
    intermediate_size: int = 2048; max_position_embeddings: int = 32768
    rms_norm_eps: float = 1e-6; rope_theta: float = 1e6; dropout: float = 0.0
    hidden_act: str = 'silu'; flash_attn: bool = True; bos_token_id: int = 1; eos_token_id: int = 2

# --- RMSNorm ---
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def norm(self, x): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    def forward(self, x): return (self.weight * self.norm(x.float())).type_as(x)

# --- RoPE ---
def precompute_freqs_cis(dim, end=32768, rope_base=1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))
    t = torch.arange(end); freqs = torch.outer(t, freqs).float()
    return torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1), torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    def rotate_half(x): return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)
    cos, sin = cos.unsqueeze(unsqueeze_dim), sin.unsqueeze(unsqueeze_dim)
    return ((q * cos) + (rotate_half(q) * sin)).to(q.dtype), ((k * cos) + (rotate_half(k) * sin)).to(k.dtype)

# --- Attention ---
def repeat_kv(x, n_rep):
    bs, slen, n_kv_heads, head_dim = x.shape
    if n_rep == 1: return x
    return x[:, :, :, None, :].expand(bs, slen, n_kv_heads, n_rep, head_dim).reshape(bs, slen, n_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout
        self.flash = hasattr(F, 'scaled_dot_product_attention') and config.flash_attn
    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        if past_key_value is not None:
            xk = torch.cat([past_key_value[0], xk], dim=1)
            xv = torch.cat([past_key_value[1], xv], dim=1)
        past_kv = (xk, xv) if use_cache else None
        xq, xk, xv = xq.transpose(1, 2), repeat_kv(xk, self.n_rep).transpose(1, 2), repeat_kv(xv, self.n_rep).transpose(1, 2)
        if self.flash and seq_len > 1 and past_key_value is None:
            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
            if attention_mask is not None:
                scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9
            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv
        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)
        return self.resid_dropout(self.o_proj(output)), past_kv

# --- FeedForward (SwiGLU) ---
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
    def forward(self, x): return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# --- Transformer Block ---
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config)
    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        residual = hidden_states
        hidden_states, present_kv = self.self_attn(self.input_layernorm(hidden_states), position_embeddings, past_key_value, use_cache, attention_mask)
        hidden_states = hidden_states + residual
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states, present_kv

# --- Full Model ---
class MiniMindModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=config.max_position_embeddings, rope_base=config.rope_theta)
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)
    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False):
        batch_size, seq_length = input_ids.shape
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0
        hidden_states = self.dropout(self.embed_tokens(input_ids))
        position_embeddings = (self.freqs_cos[start_pos:start_pos + seq_length], self.freqs_sin[start_pos:start_pos + seq_length])
        presents = []
        for layer, past_kv in zip(self.layers, past_key_values):
            hidden_states, present = layer(hidden_states, position_embeddings, past_key_value=past_kv, use_cache=use_cache, attention_mask=attention_mask)
            presents.append(present)
        return self.norm(hidden_states), presents

class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = MiniMindModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.model.embed_tokens.weight = self.lm_head.weight
    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, use_cache=False):
        hidden_states, past_key_values = self.model(input_ids, attention_mask, past_key_values, use_cache)
        logits = self.lm_head(hidden_states)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)
        return {'loss': loss, 'logits': logits, 'past_key_values': past_key_values}

# --- Synthetic Data ---
pretrain_texts = [
    "The transformer architecture was introduced in 2017.",
    "Language models learn to predict the next token in a sequence.",
    "Attention mechanisms allow models to focus on relevant parts of the input.",
    "Neural networks consist of layers of interconnected nodes.",
    "Gradient descent is the primary optimization algorithm for training neural networks.",
    "Backpropagation computes gradients by applying the chain rule.",
    "Tokenization converts raw text into numerical representations.",
    "The softmax function converts logits into probability distributions.",
    "Residual connections help train very deep networks.",
    "Layer normalization stabilizes training by normalizing activations.",
    "Dropout randomly zeroes activations to prevent overfitting.",
    "The learning rate controls the step size during optimization.",
    "Batch size determines how many samples are processed together.",
    "Cross-entropy loss measures the difference between predicted and true distributions.",
    "Weight decay adds L2 regularization to prevent overfitting.",
    "Mixed precision training uses float16 to speed up computation.",
    "The Adam optimizer combines momentum and adaptive learning rates.",
    "Cosine annealing gradually reduces the learning rate during training.",
    "Embedding layers map discrete tokens to continuous vectors.",
    "Position embeddings encode the order of tokens in a sequence.",
]
os.makedirs('synthetic_data', exist_ok=True)
pretrain_path = 'synthetic_data/pretrain.jsonl'
with open(pretrain_path, 'w') as f:
    for text in pretrain_texts:
        f.write(json.dumps({'text': text}) + '\n')

sft_conversations = [
    [{"role": "user", "content": "What is a transformer?"}, {"role": "assistant", "content": "A transformer is a neural network architecture that uses self-attention mechanisms to process sequential data in parallel, rather than sequentially like RNNs."}],
    [{"role": "user", "content": "Explain attention in neural networks."}, {"role": "assistant", "content": "Attention allows a model to dynamically focus on different parts of the input when producing each output element. It computes weighted sums of values based on query-key similarity scores."}],
    [{"role": "user", "content": "What is tokenization?"}, {"role": "assistant", "content": "Tokenization is the process of breaking text into smaller units called tokens. Common methods include BPE (Byte-Pair Encoding), which iteratively merges frequent character pairs."}],
    [{"role": "user", "content": "How does gradient descent work?"}, {"role": "assistant", "content": "Gradient descent updates model parameters by moving them in the direction that reduces the loss function. The step size is controlled by the learning rate."}],
    [{"role": "user", "content": "What is overfitting?"}, {"role": "assistant", "content": "Overfitting occurs when a model learns the training data too well, including noise and outliers, and fails to generalize to new unseen data. Techniques like dropout and weight decay help prevent it."}],
    [{"role": "user", "content": "Explain the softmax function."}, {"role": "assistant", "content": "Softmax converts a vector of real numbers into a probability distribution. Each output value is between 0 and 1, and all outputs sum to 1. It is commonly used in the final layer of classification models."}],
    [{"role": "user", "content": "What is transfer learning?"}, {"role": "assistant", "content": "Transfer learning involves taking a model trained on one task and adapting it to a different but related task. This saves computation and often improves performance, especially with limited data."}],
    [{"role": "user", "content": "What is a loss function?"}, {"role": "assistant", "content": "A loss function measures how far the model's predictions are from the true values. Common examples include cross-entropy for classification and mean squared error for regression."}],
]
sft_path = 'synthetic_data/sft.jsonl'
with open(sft_path, 'w') as f:
    for conv in sft_conversations:
        f.write(json.dumps({'conversations': conv}) + '\n')

# --- Tokenizer (BPE) ---
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer
VOCAB_SIZE = 6400
all_special = [
    '<|endoftext|>', '<|im_start|>', '<|im_end|>', '<|im_sep|>',
    '<|endofprefix|>', '<|startoftext|>', '<|extra_0|>', '<|extra_1|>',
    '<|extra_2|>', '<|extra_3|>', '<|extra_4|>', '<|extra_5|>',
    '<|extra_6|>', '<|extra_7|>', '<|extra_8|>', '<|extra_9|>',
    '<pad>', '<mask>', '<sep>', '<cls>',
    '<unused_0>', '<unused_1>', '<unused_2>', '<unused_3>',
    '<unused_4>', '<unused_5>', '<unused_6>', '<unused_7>',
    '<think>', '</think>', '<tool_call>', '</tool_call>',
    '<tool_response>', '</tool_response>', '<code>', '</code>',
]
with open(pretrain_path) as f:
    corpus = [json.loads(line)['text'] for line in f] * 100
raw_tokenizer = Tokenizer(models.BPE())
raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer_obj = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=all_special, show_progress=False, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
raw_tokenizer.decoder = decoders.ByteLevel()
raw_tokenizer.train_from_iterator(corpus, trainer=trainer_obj)

class SimpleTokenizer:
    def __init__(self, tokenizer, special_tokens):
        self.tokenizer = tokenizer
        vocab = tokenizer.get_vocab()
        self.bos_token_id = vocab.get('<|im_start|>', 1)
        self.eos_token_id = vocab.get('<|im_end|>', 2)
        self.pad_token_id = vocab.get('<pad>', 0)
    def __call__(self, text, add_special_tokens=False, max_length=None, truncation=False):
        ids = self.tokenizer.encode(text).ids
        if max_length and truncation: ids = ids[:max_length]
        return type('Encoding', (), {'input_ids': ids})()
    def decode(self, ids): return self.tokenizer.decode(ids)
tokenizer = SimpleTokenizer(raw_tokenizer, all_special)

# --- Chat Template ---
def apply_chat_template(messages):
    text = ''
    for msg in messages:
        text += f'<|im_start|>{msg["role"]}\n{msg["content"]}<|im_end|>\n'
    return text

# --- Datasets ---
class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=128):
        super().__init__(); self.tokenizer = tokenizer; self.max_length = max_length; self.samples = []
        with open(data_path) as f:
            for line in f: self.samples.append(json.loads(line))
    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        text = str(self.samples[index]['text'])
        tokens = self.tokenizer(text, add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        input_ids = torch.tensor(input_ids[:self.max_length], dtype=torch.long)
        labels = input_ids.clone(); labels[input_ids == self.tokenizer.pad_token_id] = -100
        return input_ids, labels

class SFTDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=256):
        super().__init__(); self.tokenizer = tokenizer; self.max_length = max_length; self.samples = []
        with open(jsonl_path) as f:
            for line in f: self.samples.append(json.loads(line))
        self.bos_marker = tokenizer(f'<|im_start|>assistant\n', add_special_tokens=False).input_ids
        self.eos_marker = tokenizer(f'<|im_end|>\n', add_special_tokens=False).input_ids
    def __len__(self): return len(self.samples)
    def generate_labels(self, input_ids):
        labels = [-100] * len(input_ids)
        i = 0
        while i < len(input_ids):
            if input_ids[i:i + len(self.bos_marker)] == self.bos_marker:
                start = i + len(self.bos_marker); end = start
                while end < len(input_ids):
                    if input_ids[end:end + len(self.eos_marker)] == self.eos_marker: break
                    end += 1
                for j in range(start, min(end + len(self.eos_marker), len(input_ids))): labels[j] = input_ids[j]
                i = end + len(self.eos_marker) if end < len(input_ids) else len(input_ids)
            else: i += 1
        return labels
    def __getitem__(self, index):
        conversations = self.samples[index]['conversations']
        prompt = apply_chat_template(conversations)
        input_ids = self.tokenizer(prompt).input_ids[:self.max_length]
        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))
        labels = self.generate_labels(input_ids)
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

# --- LR Schedule & Generate ---
def get_lr(step, total_steps, warmup_steps, max_lr, min_lr=1e-6):
    if step < warmup_steps: return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def generate(model, tokenizer, prompt_ids, max_new_tokens=50, temperature=1.0, top_k=0):
    model.eval()
    input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(device)
    for _ in range(max_new_tokens):
        outputs = model(input_ids)
        logits = outputs['logits'][:, -1, :] / temperature
        if top_k > 0:
            topk_values, _ = torch.topk(logits, top_k)
            logits[logits < topk_values[:, -1:]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        if next_token.item() == tokenizer.eos_token_id: break
    return input_ids[0].tolist()

# --- LoRA ---
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank=16):
        super().__init__()
        self.rank = rank
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        self.A.weight.data.normal_(mean=0.0, std=0.02)
        self.B.weight.data.zero_()
    def forward(self, x): return self.B(self.A(x))

def apply_lora(model, rank=16):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.weight.shape[0] == module.weight.shape[1]:
            lora = LoRA(module.weight.shape[0], module.weight.shape[1], rank=rank).to(next(model.parameters()).device)
            setattr(module, 'lora', lora)
            original_forward = module.forward
            def forward_with_lora(x, layer1=original_forward, layer2=lora): return layer1(x) + layer2(x)
            module.forward = forward_with_lora
    return model

def save_lora(model, path):
    state_dict = {}
    for name, module in model.named_modules():
        if hasattr(module, 'lora'):
            lora_state = {f'{name}.lora.{k}': v.cpu() for k, v in module.lora.state_dict().items()}
            state_dict.update(lora_state)
    torch.save(state_dict, path)
    return len(state_dict)

def load_lora(model, path):
    state_dict = torch.load(path, map_location=next(model.parameters()).device, weights_only=True)
    for name, module in model.named_modules():
        if hasattr(module, 'lora'):
            lora_state = {k.replace(f'{name}.lora.', ''): v for k, v in state_dict.items() if f'{name}.lora.' in k}
            module.lora.load_state_dict(lora_state)

def merge_lora(model, lora_path, save_path):
    load_lora(model, lora_path)
    state_dict = {k: v.cpu() for k, v in model.state_dict().items() if '.lora.' not in k}
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and hasattr(module, 'lora'):
            state_dict[f'{name}.weight'] = (
                module.weight.data.clone().cpu() +
                (module.lora.B.weight.data @ module.lora.A.weight.data).cpu()
            )
    torch.save(state_dict, save_path)

# --- Quick Pretrain Run ---
config = MiniMindConfig()
pretrain_ds = PretrainDataset(pretrain_path, tokenizer, max_length=64)
sft_ds = SFTDataset(sft_path, tokenizer, max_length=128)
model = MiniMindForCausalLM(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
train_loader = DataLoader(pretrain_ds, batch_size=4, shuffle=True)
total_steps = 3 * len(train_loader); warmup_steps = int(total_steps * 0.1)
CHECKPOINT_DIR = 'minimind_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model.train()
for epoch in range(3):
    for step, (input_ids, labels) in enumerate(train_loader):
        input_ids, labels = input_ids.to(device), labels.to(device)
        lr = get_lr(epoch * len(train_loader) + step, total_steps, warmup_steps, 5e-4)
        for pg in optimizer.param_groups: pg['lr'] = lr
        loss = model(input_ids, labels=labels)['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad()
    print(f'  Pretrain epoch {epoch+1}/3 — loss: {loss.item():.4f}')
pretrain_ckpt = os.path.join(CHECKPOINT_DIR, 'pretrain_final.pth')
torch.save(model.state_dict(), pretrain_ckpt)
del model, optimizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- Quick SFT Run ---
sft_model = MiniMindForCausalLM(config).to(device)
sft_model.load_state_dict(torch.load(pretrain_ckpt, map_location=device, weights_only=True))
sft_optimizer = torch.optim.AdamW(sft_model.parameters(), lr=1e-4, weight_decay=0.01)
sft_loader = DataLoader(sft_ds, batch_size=4, shuffle=True)
total_sft_steps = 3 * len(sft_loader); warmup_sft = int(total_sft_steps * 0.1)
sft_model.train()
for epoch in range(3):
    for step, (input_ids, labels) in enumerate(sft_loader):
        input_ids, labels = input_ids.to(device), labels.to(device)
        lr = get_lr(epoch * len(sft_loader) + step, total_sft_steps, warmup_sft, 1e-4)
        for pg in sft_optimizer.param_groups: pg['lr'] = lr
        loss = sft_model(input_ids, labels=labels)['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
        sft_optimizer.step(); sft_optimizer.zero_grad()
    print(f'  SFT epoch {epoch+1}/3 — loss: {loss.item():.4f}')
sft_ckpt = os.path.join(CHECKPOINT_DIR, 'full_sft.pth')
torch.save(sft_model.state_dict(), sft_ckpt)
del sft_model, sft_optimizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- Quick LoRA Run ---
lora_model = MiniMindForCausalLM(config).to(device)
lora_model.load_state_dict(torch.load(pretrain_ckpt, map_location=device, weights_only=True))
apply_lora(lora_model, rank=16)
for name, param in lora_model.named_parameters():
    if 'lora' not in name: param.requires_grad = False
lora_optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.01)
lora_loader = DataLoader(sft_ds, batch_size=4, shuffle=True)
total_lora_steps = 3 * len(lora_loader); warmup_lora = int(total_lora_steps * 0.1)
lora_model.train()
for epoch in range(3):
    for step, (input_ids, labels) in enumerate(lora_loader):
        input_ids, labels = input_ids.to(device), labels.to(device)
        lr = get_lr(epoch * len(lora_loader) + step, total_lora_steps, warmup_lora, 1e-4)
        for pg in lora_optimizer.param_groups: pg['lr'] = lr
        loss = lora_model(input_ids, labels=labels)['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)
        lora_optimizer.step(); lora_optimizer.zero_grad()
    print(f'  LoRA epoch {epoch+1}/3 — loss: {loss.item():.4f}')
lora_save_path = os.path.join(CHECKPOINT_DIR, 'lora_weights.pth')
save_lora(lora_model, lora_save_path)
del lora_model, lora_optimizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- Merge LoRA ---
merge_model = MiniMindForCausalLM(config).to(device)
merge_model.load_state_dict(torch.load(pretrain_ckpt, map_location=device, weights_only=True))
apply_lora(merge_model, rank=16)
merged_path = os.path.join(CHECKPOINT_DIR, 'merged_model.pth')
merge_lora(merge_model, lora_save_path, merged_path)
del merge_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\n✅ All prior code loaded — config: {config.hidden_size}d, {config.num_hidden_layers}L, {config.vocab_size}V')
print(f'   Tokenizer: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}')
print(f'   Pretrain: {len(pretrain_ds)} samples, SFT: {len(sft_ds)} samples')
print(f'   Checkpoints: pretrain={pretrain_ckpt}, sft={sft_ckpt}')
print(f'   LoRA: {lora_save_path}, Merged: {merged_path}')

## Part A — Decoding Strategies

When generating text, the model outputs a probability distribution over the vocabulary at each step. How we select the next token from this distribution matters greatly:

| Strategy | Description |
|----------|-------------|
| **Greedy** | Always pick the highest probability token |
| **Random Sampling** | Sample from the full distribution |
| **Top-k** | Sample from the k most likely tokens |
| **Top-p (Nucleus)** | Sample from the smallest set whose cumulative probability ≥ p |
| **Temperature** | Scale logits before softmax: higher = more random, lower = more deterministic |

In [ ]:
# === Advanced Generation with All Strategies ===
@torch.no_grad()
def generate_advanced(model, tokenizer, prompt_ids, max_new_tokens=50,
                      temperature=1.0, top_k=0, top_p=1.0,
                      repetition_penalty=1.0, do_sample=True):
    """Generate text with various decoding strategies.
    
    Args:
        temperature: Scale logits. <1 = focused, >1 = diverse, 1 = neutral
        top_k: Keep only top-k tokens (0 = disabled)
        top_p: Keep smallest set with cumulative prob >= p (1.0 = disabled)
        repetition_penalty: Penalize repeated tokens (1.0 = disabled)
        do_sample: If False, use greedy decoding
    """
    model.eval()
    input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(device)
    
    for _ in range(max_new_tokens):
        outputs = model(input_ids)
        logits = outputs['logits'][:, -1, :]
        
        # Repetition penalty
        if repetition_penalty != 1.0:
            for token_id in set(input_ids[0].tolist()):
                logits[0, token_id] /= repetition_penalty
        
        # Temperature
        logits = logits / temperature
        
        if not do_sample:
            # Greedy decoding
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            # Top-k filtering
            if top_k > 0:
                topk_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < topk_values[:, -1:]] = float('-inf')
            
            # Top-p (nucleus) filtering
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
                sorted_logits[mask] = float('-inf')
                logits = sorted_logits.scatter(1, sorted_indices, sorted_logits)
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return input_ids[0].tolist()

print('Advanced generate function defined ✓')

In [ ]:
# === Compare Decoding Strategies ===
# Load the SFT model for comparison
compare_model = MiniMindForCausalLM(config).to(device)
compare_model.load_state_dict(torch.load(sft_ckpt, map_location=device, weights_only=True))
compare_model.eval()

prompt_text = 'The'
prompt_ids = tokenizer(prompt_text).input_ids

strategies = {
    'Greedy':          dict(do_sample=False),
    'Sampling T=0.5':  dict(temperature=0.5),
    'Sampling T=1.0':  dict(temperature=1.0),
    'Sampling T=1.5':  dict(temperature=1.5),
    'Top-k (k=10)':    dict(top_k=10),
    'Top-k (k=50)':    dict(top_k=50),
    'Top-p (p=0.9)':   dict(top_p=0.9),
    'Top-p (p=0.5)':   dict(top_p=0.5),
    'Combined':        dict(temperature=0.8, top_k=50, top_p=0.9, repetition_penalty=1.2),
}

print(f'=== Decoding Strategy Comparison ===')
print(f'Prompt: "{prompt_text}"\n')

for name, kwargs in strategies.items():
    gen_ids = generate_advanced(compare_model, tokenizer, prompt_ids, max_new_tokens=30, **kwargs)
    text = raw_tokenizer.decode(gen_ids)
    print(f'{name:20s} → {text}')

In [ ]:
# === ✅ Milestone 6A: Decoding Strategies ===
# Greedy should always produce the same output
greedy1 = generate_advanced(compare_model, tokenizer, prompt_ids, max_new_tokens=20, do_sample=False)
greedy2 = generate_advanced(compare_model, tokenizer, prompt_ids, max_new_tokens=20, do_sample=False)
assert greedy1 == greedy2, 'Greedy decoding should be deterministic!'

# Sampling should (usually) produce different outputs
samples = set()
for _ in range(5):
    gen = generate_advanced(compare_model, tokenizer, prompt_ids, max_new_tokens=20, temperature=1.0)
    samples.add(tuple(gen))

print(f'✅ Milestone 6A passed — decoding strategies work')
print(f'   Greedy is deterministic: ✓')
print(f'   Sampling produced {len(samples)} unique outputs from 5 runs')

## Part B — Temperature Exploration

Temperature controls the "sharpness" of the probability distribution:
- **T → 0**: Distribution becomes one-hot (greedy) — always picks the most likely token
- **T = 1**: Original distribution — balanced exploration
- **T → ∞**: Distribution becomes uniform — maximum randomness

$$P(x_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

In [ ]:
# === Temperature Effect Visualization ===
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Get logits for a sample input
sample_input = torch.randint(0, config.vocab_size, (1, 10)).to(device)
with torch.no_grad():
    logits = compare_model(sample_input)['logits'][0, -1, :].cpu()

temperatures = [0.1, 0.5, 1.0, 1.5, 3.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(16, 3))

for ax, temp in zip(axes, temperatures):
    probs = F.softmax(logits / temp, dim=-1).numpy()
    # Show top 20 tokens
    top_k_idx = np.argsort(probs)[-20:][::-1]
    top_probs = probs[top_k_idx]
    ax.bar(range(len(top_probs)), top_probs, color='steelblue')
    ax.set_title(f'T = {temp}')
    ax.set_xlabel('Token rank')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, max(0.1, top_probs[0] * 1.2))

plt.suptitle('Effect of Temperature on Token Probability Distribution', y=1.02)
plt.tight_layout()
plt.savefig('temperature_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print('Lower temperature → more peaked (deterministic)')
print('Higher temperature → more uniform (random)')

## Part C — Perplexity Evaluation

**Perplexity** measures how well the model predicts a text sequence:

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(x_i | x_{<i})\right)$$

- Lower perplexity = better predictions
- A perplexity of V (vocab size) means the model is guessing randomly
- A perplexity of 1 means perfect prediction

In [ ]:
# === Perplexity Evaluation ===
@torch.no_grad()
def compute_perplexity(model, dataset, max_samples=None):
    """Compute perplexity on a dataset."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    n_samples = min(len(dataset), max_samples) if max_samples else len(dataset)
    
    for i in range(n_samples):
        input_ids, labels = dataset[i]
        input_ids = input_ids.unsqueeze(0).to(device)
        labels = labels.unsqueeze(0).to(device)
        
        outputs = model(input_ids, labels=labels)
        loss = outputs['loss']
        
        # Count non-masked tokens
        n_tokens = (labels != -100).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens
    
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity, avg_loss

In [ ]:
# === ✅ Milestone 6B: Model Comparison via Perplexity ===
# Load all three models
pretrain_model = MiniMindForCausalLM(config).to(device)
pretrain_model.load_state_dict(torch.load(pretrain_ckpt, map_location=device, weights_only=True))

sft_model_eval = MiniMindForCausalLM(config).to(device)
sft_model_eval.load_state_dict(torch.load(sft_ckpt, map_location=device, weights_only=True))

merged_model = MiniMindForCausalLM(config).to(device)
merged_model.load_state_dict(torch.load(merged_path, map_location=device, weights_only=True))

random_model = MiniMindForCausalLM(config).to(device)  # Random baseline

# Evaluate on pretrain data
pretrain_eval = PretrainDataset(pretrain_path, tokenizer, max_length=64)

models_to_eval = {
    'Random (untrained)': random_model,
    'Pretrained': pretrain_model,
    'Full SFT': sft_model_eval,
    'LoRA Merged': merged_model,
}

print('=== Perplexity Comparison (on pretrain data) ===')
print(f'{"Model":<25s} {"Perplexity":>12s} {"Avg Loss":>10s}')
print('-' * 50)

results = {}
for name, m in models_to_eval.items():
    ppl, avg_loss = compute_perplexity(m, pretrain_eval, max_samples=10)
    results[name] = ppl
    print(f'{name:<25s} {ppl:>12.2f} {avg_loss:>10.4f}')

# Pretrained should have lower perplexity than random
assert results['Pretrained'] < results['Random (untrained)'], 'Pretrained should beat random!'
print(f'\n✅ Milestone 6B passed — perplexity evaluation works')
print(f'   Random baseline PPL: {results["Random (untrained)"]:.1f}')
print(f'   Best model PPL: {min(results.values()):.1f}')

## Part D — Multi-Turn Conversation

A practical LLM maintains conversation history across turns. Each new user message is appended to the history and the full context is fed to the model.

In [ ]:
# === Multi-Turn Conversation ===
def chat(model, tokenizer, messages, max_new_tokens=60, temperature=0.8, top_k=50):
    """Generate a response given conversation history."""
    # Add generation prompt for the next assistant turn
    prompt = apply_chat_template(messages)
    prompt += '<|im_start|>assistant\n'
    
    prompt_ids = tokenizer(prompt).input_ids
    gen_ids = generate_advanced(
        model, tokenizer, prompt_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        repetition_penalty=1.1
    )
    
    # Extract just the new response
    response_ids = gen_ids[len(prompt_ids):]
    response = raw_tokenizer.decode(response_ids)
    
    # Clean up: remove any trailing special tokens
    for token in ['<|im_end|>', '<|im_start|>']:
        response = response.split(token)[0]
    
    return response.strip()

# Multi-turn conversation demo
best_model = sft_model_eval  # Use SFT model

conversation = []
exchanges = [
    "What is a neural network?",
    "How does it learn?",
    "What is backpropagation?",
]

print('=== Multi-Turn Conversation ===\n')
for user_msg in exchanges:
    conversation.append({"role": "user", "content": user_msg})
    response = chat(best_model, tokenizer, conversation)
    conversation.append({"role": "assistant", "content": response})
    
    print(f'👤 User: {user_msg}')
    print(f'🤖 Assistant: {response}')
    print()

print(f'Conversation history: {len(conversation)} messages ({len(conversation)//2} turns)')

In [ ]:
# === ✅ Milestone 6C: Multi-Turn Conversation ===
test_conv = [{"role": "user", "content": "Hello"}]
response = chat(best_model, tokenizer, test_conv)

assert isinstance(response, str), 'Response should be a string'
assert len(response) > 0, 'Response should not be empty'

# Test multi-turn
test_conv.append({"role": "assistant", "content": response})
test_conv.append({"role": "user", "content": "Tell me more"})
response2 = chat(best_model, tokenizer, test_conv)

assert len(response2) > 0, 'Second response should not be empty'

print(f'✅ Milestone 6C passed — multi-turn conversation works')
print(f'   Turn 1 response: "{response[:50]}..."')
print(f'   Turn 2 response: "{response2[:50]}..."')

## Part E — Final Model Comparison

Let's do a comprehensive side-by-side comparison of all our models, testing them on the same prompts.

In [ ]:
# === Final Model Comparison ===
test_prompts = [
    [{"role": "user", "content": "What is a language model?"}],
    [{"role": "user", "content": "Explain attention."}],
    [{"role": "user", "content": "What is tokenization?"}],
]

model_names = ['Pretrained', 'Full SFT', 'LoRA Merged']
models_list = [pretrain_model, sft_model_eval, merged_model]

print('=== Side-by-Side Model Comparison ===\n')
for messages in test_prompts:
    question = messages[0]['content']
    print(f'📝 Question: {question}')
    print('-' * 60)
    for name, m in zip(model_names, models_list):
        response = chat(m, tokenizer, messages, max_new_tokens=40)
        print(f'  {name:15s}: {response[:80]}')
    print()

## 🎯 Notebook 6 Summary

| Component | Key Concept |
|---|---|
| **Greedy Decoding** | Always pick highest probability — deterministic but repetitive |
| **Top-k Sampling** | Sample from k most likely tokens — simple diversity control |
| **Top-p (Nucleus)** | Sample from dynamic token set — adaptive diversity |
| **Temperature** | Scale logits: T<1 focused, T>1 diverse |
| **Repetition Penalty** | Reduce probability of previously generated tokens |
| **Perplexity** | exp(avg cross-entropy) — lower is better |
| **Multi-Turn Chat** | Maintain conversation history across turns |

**Milestones completed:** 6A (decoding strategies), 6B (perplexity), 6C (multi-turn chat)

---

## 🏆 Complete Step-by-Step Series Summary

| Notebook | Topic | Milestones |
|----------|-------|------------|
| 1 | Tokenizer & Text Processing | 1A, 1B, 1C |
| 2 | Model Architecture | 2A, 2B, 2C |
| 3 | Data Pipeline | 3A, 3B, 3C |
| 4 | Pretraining | 4A, 4B |
| 5 | SFT & LoRA | 5A, 5B, 5C |
| 6 | Evaluation & Inference | 6A, 6B, 6C |

**Total: 17 milestones across 6 notebooks**

You've built a complete language model from scratch:
1. ✅ Trained a BPE tokenizer
2. ✅ Built a transformer decoder (RMSNorm, RoPE, GQA, SwiGLU)
3. ✅ Created data pipelines with proper label masking
4. ✅ Pretrained on text data with AMP and cosine LR
5. ✅ Fine-tuned with full SFT and parameter-efficient LoRA
6. ✅ Evaluated with perplexity and explored decoding strategies

### Going Further
- **Scale up**: Use larger datasets from HuggingFace Hub
- **DPO/RLHF**: Align the model with human preferences (see `trainer/train_dpo.py`)
- **Tool calling**: Teach the model to use tools (see `trainer/train_agent.py`)
- **Deployment**: Serve via API (see `scripts/serve_openai_api.py`) or Web UI (see `scripts/web_demo.py`)

Congratulations! 🎉